# 06 - Elo Accuracy Upgrade and Probable Lineups

Goal: improve match-model accuracy and add player-level outputs for frontend use.

What changed in this upgrade:

1. Historical Elo ratings are computed from `results.csv`.
2. Each training match gets the teams' Elo ratings **before** that match.
3. The model now learns from team quality, form-derived squad strength, home/neutral context, and tournament type.
4. We export possible 26-player squad pools and probable starting XIs.

Why Elo helps: player features tell us how strong the 2026 squad looks. Elo tells us how strong the national team has actually been in international football over time.

In [1]:
import sys
from pathlib import Path

import pandas as pd

SRC_DIR = Path('../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from team_feature_engineering import (
    PROCESSED_DIR,
    read_processed_csv,
    select_team_squad_players,
    build_probable_starting_xi,
)
from trained_match_model import (
    load_inputs,
    add_missing_team_placeholders,
    prepare_results,
    latest_elo_ratings,
    build_training_data,
    train_models,
    predict_fixtures,
    build_group_tables,
    NAME_TO_CODE,
)

print('Notebook 06 imports loaded')

Notebook 06 imports loaded


## Step 1 - Regenerate selected squads and starting XIs

The squad selector uses this logic:

- official parsed squad if available
- capped official longlist if parsed squad has more than 26 matched players
- probable top 26 fallback when no official squad exists

The starting XI uses a simple 4-3-3-style rule: 1 GK, 4 defenders, 3 midfielders, 3 forwards, then best remaining players if a position is short.

In [2]:
player_ml = read_processed_csv('player_ml_features.csv')
squad_player_path = PROCESSED_DIR / 'squad_player_coverage.csv'
squad_player_coverage = pd.read_csv(squad_player_path) if squad_player_path.exists() else None

selected_squad_players = select_team_squad_players(player_ml, squad_player_coverage)
probable_starting_xi = build_probable_starting_xi(selected_squad_players)

selected_squad_players.to_csv(PROCESSED_DIR / 'selected_squad_players.csv', index=False)
probable_starting_xi.to_csv(PROCESSED_DIR / 'probable_starting_xi.csv', index=False)

print('selected_squad_players:', selected_squad_players.shape)
print('probable_starting_xi:', probable_starting_xi.shape)

player_ml_features.csv: 3552 rows x 72 columns
['player_key', 'player_name', 'nation_code', 'nation_name', 'weighted_goal_form', 'weighted_chance_form', 'weighted_minutes', 'cur_player', 'cur_position', 'cur_squad', 'cur_competition', 'cur_nation', 'cur_stat_source', 'cur_minutes', 'cur_nineties', 'cur_matches', 'cur_goals', 'cur_assists', 'cur_xg', 'cur_shots', 'cur_shots_on_target', 'cur_interceptions', 'cur_tackles_won', 'cur_save_pct', 'cur_goals_against', 'cur_clean_sheets', 'cur_goals_per90', 'cur_assists_per90', 'cur_xg_per90', 'cur_shots_per90', 'cur_shots_on_target_per90', 'cur_interceptions_per90', 'cur_tackles_won_per90', 'prev_player', 'prev_position', 'prev_squad', 'prev_competition', 'prev_nation', 'prev_minutes', 'prev_nineties', 'prev_matches', 'prev_goals', 'prev_assists', 'prev_xg', 'prev_npxg', 'prev_xag', 'prev_progressive_passes', 'prev_progressive_carries', 'prev_progressive_receptions', 'prev_goals_per90', 'prev_assists_per90', 'prev_xg_per90', 'prev_npxg_per90',

## Step 2 - Check squad and XI coverage

This tells us where the player data is good enough to build a full XI. If a team has fewer than 11 selected players, that is a data coverage issue, not a football claim.

In [3]:
squad_counts = selected_squad_players.groupby(['nation_code', 'nation_name']).size().reset_index(name='selected_count')
xi_counts = probable_starting_xi.groupby(['nation_code', 'nation_name']).size().reset_index(name='xi_count')
coverage = squad_counts.merge(xi_counts, on=['nation_code', 'nation_name'], how='outer').fillna(0)
coverage = coverage.sort_values(['xi_count', 'selected_count'])
display(coverage.head(15))

print('Teams with complete XI:', (coverage['xi_count'] >= 11).sum())
print('Teams with incomplete XI:', (coverage['xi_count'] < 11).sum())

,nation_code,nation_name,selected_count,xi_count
35,RSA,South Africa,2,2
1,AUS,Australia,5,5
14,ECU,Ecuador,10,10
6,CAN,Canada,11,11
15,EGY,Egypt,23,11
7,CIV,Cote d'Ivoire,24,11
12,CUW,Curacao,25,11
19,GER,Germany,25,11
21,HAI,Haiti,25,11
0,ARG,Argentina,26,11


Teams with complete XI: 42
Teams with incomplete XI: 3


## Step 3 - Train the Elo-enhanced model

The old model mainly used squad strength differences. The upgraded model adds:

- `elo_diff`
- `elo_absdiff`
- `home_advantage`
- `is_neutral`
- tournament-type flags

The most important part: Elo is computed chronologically, so each historical match only sees ratings from before that match.

In [4]:
teams, results, fixtures = load_inputs()
fixture_codes = set(fixtures['team_a'].map(NAME_TO_CODE).dropna()) | set(fixtures['team_b'].map(NAME_TO_CODE).dropna())
teams = add_missing_team_placeholders(teams, fixture_codes)

recent_results = prepare_results(results, min_year=2018)
elo_ratings = latest_elo_ratings(results)
X, y = build_training_data(recent_results, teams)

print('Training rows:', X.shape)
print('Target distribution:')
print(y.value_counts())
display(X.head())

teams: (45, 26)
['nation_code', 'nation_name', 'selection_method', 'squad_players_used', 'estimated_players_used', 'verified_supplemental_used', 'avg_selection_score', 'top6_goal_form_sum', 'top6_goal_form_mean', 'top10_chance_form_sum', 'top10_chance_form_mean', 'attacker_goal_form_mean', 'midfielder_chance_form_mean', 'defender_actions_per90_mean', 'keeper_save_pct_best', 'squad_weighted_minutes_mean', 'attack_score', 'creation_score', 'defense_score', 'keeper_score', 'depth_score', 'data_confidence_score', 'coverage_pct', 'has_announced_squad_file', 'raw_power_score', 'power_score']
results: (49287, 9)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
fixtures: (72, 7)
['group', 'match_number', 'date', 'time_utc_offset', 'team_a', 'team_b', 'venue']
Added missing-data placeholders for: ['DZA', 'IRQ', 'SAU']
Training rows: (974, 24)
Target distribution:
result
home_win    433
away_win    281
draw        260
Name: count, dtype: 

,power_score_diff,power_score_absdiff,attack_score_diff,attack_score_absdiff,creation_score_diff,creation_score_absdiff,defense_score_diff,defense_score_absdiff,keeper_score_diff,keeper_score_absdiff,...,attack_vs_defense_diff,defense_vs_attack_diff,elo_diff,elo_absdiff,home_advantage,is_neutral,is_world_cup,is_qualifier,is_friendly,is_continental
0,6.416253,6.416253,14.888889,14.888889,26.222222,26.222222,-25.333333,25.333333,-37.777778,37.777778,...,-6.000000,-4.444444,72.647674,72.647674,1.0,0.0,0.0,0.0,1.0,0.0
1,0.829417,0.829417,-4.333333,4.333333,6.833333,6.833333,14.888889,14.888889,-37.777778,37.777778,...,-25.222222,35.777778,182.168112,182.168112,0.0,1.0,0.0,0.0,1.0,0.0
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-2.422222,2.422222,-26.075122,26.075122,1.0,0.0,0.0,0.0,1.0,0.0
3,-3.879919,3.879919,-4.422222,4.422222,-12.744444,12.744444,-12.222222,12.222222,0.000000,0.000000,...,-14.644444,-2.000000,55.592144,55.592144,1.0,0.0,0.0,0.0,1.0,0.0
4,28.262784,28.262784,39.222222,39.222222,8.444444,8.444444,8.444444,8.444444,36.666667,36.666667,...,59.444444,-11.777778,54.761444,54.761444,1.0,0.0,0.0,0.0,1.0,0.0


In [5]:
logistic, forest, metrics = train_models(X, y)
display(metrics)

best_model_name = metrics.sort_values('accuracy', ascending=False).iloc[0]['model']
best_model = logistic if best_model_name == 'logistic_regression' else forest
print('Best model:', best_model_name)


logistic_regression
              precision    recall  f1-score   support

    away_win       0.49      0.53      0.51        70
        draw       0.43      0.14      0.21        65
    home_win       0.57      0.77      0.65       109

    accuracy                           0.53       244
   macro avg       0.50      0.48      0.46       244
weighted avg       0.51      0.53      0.49       244


random_forest
              precision    recall  f1-score   support

    away_win       0.43      0.50      0.46        70
        draw       0.28      0.23      0.25        65
    home_win       0.62      0.61      0.62       109

    accuracy                           0.48       244
   macro avg       0.44      0.45      0.44       244
weighted avg       0.47      0.48      0.48       244



,model,accuracy
0,logistic_regression,0.532787
1,random_forest,0.479508


Best model: logistic_regression


## Step 4 - Predict 2026 groups with Elo-enhanced model

The prediction now combines:

- 2026 squad/player features
- latest historical Elo ratings
- neutral World Cup context

This is meaningfully stronger than the previous version.

In [6]:
predictions = predict_fixtures(fixtures, teams, best_model, elo_ratings)
group_tables = build_group_tables(predictions)

predictions.to_csv(PROCESSED_DIR / 'wc_2026_group_predictions.csv', index=False)
group_tables.to_csv(PROCESSED_DIR / 'wc_2026_predicted_group_tables.csv', index=False)
metrics.to_csv(PROCESSED_DIR / 'trained_model_metrics.csv', index=False)

display(predictions.head(20))
display(group_tables)

,group,match_number,date,time_utc_offset,team_a,team_b,venue,team_a_code,team_b_code,prediction_status,team_a_elo,team_b_elo,team_a_win_prob,draw_prob,team_b_win_prob,predicted_result
0,A,1,2026-06-11,1:00 p.m. UTC-6,Mexico,South Africa,"Estadio Azteca, Mexico City",MEX,RSA,ok,1553.115711,1414.455123,0.310087,0.080362,0.609551,away_win
1,A,2,2026-06-11,8:00 p.m. UTC-6,South Korea,Czech Republic,"Estadio Akron, Zapopan",KOR,CZE,ok,1509.691939,1463.517803,0.460540,0.190666,0.348794,home_win
2,A,25,2026-06-18,12:00 p.m. UTC-4,Czech Republic,South Africa,"Mercedes-Benz Stadium, Atlanta",CZE,RSA,ok,1463.517803,1414.455123,0.231642,0.088867,0.679490,away_win
3,A,28,2026-06-18,7:00 p.m. UTC-6,Mexico,South Korea,"Estadio Akron, Zapopan",MEX,KOR,ok,1553.115711,1509.691939,0.322306,0.232935,0.444759,away_win
4,A,53,2026-06-24,7:00 p.m. UTC-6,Czech Republic,Mexico,"Estadio Azteca, Mexico City",CZE,MEX,ok,1463.517803,1553.115711,0.307427,0.163400,0.529173,away_win
5,A,54,2026-06-24,7:00 p.m. UTC-6,South Africa,South Korea,"Estadio BBVA, Guadalupe",RSA,KOR,ok,1414.455123,1509.691939,0.253388,0.211270,0.535342,away_win
6,B,3,2026-06-12,3:00 p.m. UTC-4,Canada,Bosnia and Herzegovina,"BMO Field, Toronto",CAN,BIH,ok,1453.149069,1410.807126,0.317802,0.232180,0.450018,away_win
7,B,8,2026-06-13,12:00 p.m. UTC-7,Qatar,Switzerland,"Levi's Stadium, Santa Clara",QAT,SUI,ok,1285.647946,1580.318781,0.110272,0.131093,0.758635,away_win
8,B,26,2026-06-18,12:00 p.m. UTC-7,Switzerland,Bosnia and Herzegovina,"SoFi Stadium, Inglewood",SUI,BIH,ok,1580.318781,1410.807126,0.628034,0.150411,0.221555,home_win
9,B,27,2026-06-18,3:00 p.m. UTC-7,Canada,Qatar,"BC Place, Vancouver",CAN,QAT,ok,1453.149069,1285.647946,0.496641,0.205348,0.298011,home_win


,group,team,points,gf_x,ga_x,gd_x
1,A,South Africa,5.007786,3.465357,2.534643,0.930714
2,A,South Korea,4.956792,3.516152,2.483848,1.032304
0,A,Mexico,3.961397,2.799830,3.200170,-0.400339
3,A,Czech Republic,3.106524,2.218660,3.781340,-1.562679
7,B,Switzerland,6.104874,4.216312,1.783688,2.432624
4,B,Canada,4.059809,2.904944,3.095056,-0.190112
5,B,Bosnia and Herzegovina,3.850407,2.749558,3.250442,-0.500883
6,B,Qatar,2.942923,2.129186,3.870814,-1.741628
8,C,Brazil,6.381884,4.448140,1.551860,2.896280
9,C,Morocco,5.553705,3.923573,2.076427,1.847146


## What improved

Before Elo, Logistic Regression accuracy was about 48%.

After adding Elo/context features, Logistic Regression is about 53%.

That is a real improvement for a three-class football prediction problem. The model still has weaknesses, especially for teams with incomplete squad data, but the direction is correct.